In [1]:
import pandas as pd

import lotus
from lotus.ast import LazyFrame
from lotus.models import LM

/Users/harshitgupta/Documents/lotus/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------------------------
# Configure the LM
# ------------------------------------------------------------------
lm = LM(model="gpt-4.1", max_batch_size=10)
lotus.settings.configure(lm=lm)

In [3]:
# ------------------------------------------------------------------
# Define the data
# ------------------------------------------------------------------
print("=" * 70)
print("Step 0: Setting up data sources")
print("=" * 70)

queries_df = pd.DataFrame(
    {
        "query_id": [1, 2],
        "query": [
            "What are the health benefits of exercise and what types of exercise are best for beginners?",
            "How does climate change affect biodiversity and what can be done to protect endangered species?",
        ],
    }
)
print("\nQueries:")
print(queries_df)

corpus_df = pd.DataFrame(
    {
        "doc_id": range(1, 11),
        "content": [
            # Health and exercise documents
            "Regular exercise improves cardiovascular health by strengthening the heart muscle and improving blood circulation. It reduces the risk of heart disease and stroke.",
            "Walking is an excellent low-impact exercise for beginners. It requires no special equipment and can be done anywhere. Start with 15-20 minutes daily.",
            "Strength training helps build muscle mass and increases metabolism. Beginners should start with bodyweight exercises like squats and push-ups.",
            "Exercise releases endorphins, which are natural mood boosters. Regular physical activity can reduce symptoms of depression and anxiety.",
            "Swimming is a great full-body workout that's easy on the joints. It's ideal for beginners and those with mobility issues.",
            # Climate and biodiversity documents
            "Climate change is causing rising temperatures that disrupt ecosystems worldwide. Many species cannot adapt quickly enough to survive.",
            "Habitat loss due to climate change is a major threat to biodiversity. Polar bears, coral reefs, and rainforest species are particularly vulnerable.",
            "Conservation efforts like protected areas and wildlife corridors help endangered species survive climate impacts.",
            "Reducing carbon emissions is crucial for protecting biodiversity. Transitioning to renewable energy can slow climate change effects.",
            "Community-based conservation programs involve local people in protecting endangered species and their habitats.",
        ],
    }
)


print("\nDocument corpus:")
print(corpus_df)

Step 0: Setting up data sources

Queries:
   query_id                                              query
0         1  What are the health benefits of exercise and w...
1         2  How does climate change affect biodiversity an...

Document corpus:
   doc_id                                            content
0       1  Regular exercise improves cardiovascular healt...
1       2  Walking is an excellent low-impact exercise fo...
2       3  Strength training helps build muscle mass and ...
3       4  Exercise releases endorphins, which are natura...
4       5  Swimming is a great full-body workout that's e...
5       6  Climate change is causing rising temperatures ...
6       7  Habitat loss due to climate change is a major ...
7       8  Conservation efforts like protected areas and ...
8       9  Reducing carbon emissions is crucial for prote...
9      10  Community-based conservation programs involve ...


In [4]:
print("\n" + "=" * 70)
print("Step 1: Building LazyFrames (lazy - no execution yet)")
print("=" * 70)

# LazyFrame 1: Query decomposition
# Split queries into subqueries, then explode into separate rows
query_df = LazyFrame("queries", queries_df).sem_map(
    "Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. "
    "Return ONLY a newline-separated list of subqueries, nothing else. "
    "Each subquery should be on its own line.",
    suffix="_subqueries",
)

print("\nQuery LazyFrame:")
print(query_df)

# LazyFrame 2: Corpus (just a source)
corpus_lf = LazyFrame("corpus", df=corpus_df)

print("\nCorpus LazyFrame:")
print(corpus_lf)


Step 1: Building LazyFrames (lazy - no execution yet)

Query LazyFrame:
Pipeline('queries')
.sem_map("Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. Return ONLY a newline-separated list of subqueries, nothing else. Each subquery should be on its own line.")

Corpus LazyFrame:
Pipeline('corpus')


In [5]:
print("\n" + "=" * 70)
print("Step 2: Explode subqueries into separate rows")
print("=" * 70)

# Process subqueries: split, explode, and clean
query_df["subquery_list"] = query_df["_subqueries"].map(lambda x: x.strip().split("\n"))
exploded_df = query_df.explode("subquery_list").reset_index(drop=True)
exploded_df = exploded_df.rename(columns={"subquery_list": "subquery"})
exploded_df["subquery"] = exploded_df["subquery"].map(lambda x: x.strip())
exploded_df = exploded_df[["query_id", "query", "subquery"]].reset_index(drop=True)

print("\nExploded subqueries LazyFrame:")
print(exploded_df)


Step 2: Explode subqueries into separate rows

Exploded subqueries LazyFrame:
Pipeline('queries')
.sem_map("Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. Return ONLY a newline-separated list of subqueries, nothing else. Each subquery should be on its own line.")
['subquery_list'] = ...
.explode('subquery_list')
.reset_index(drop=True)
.rename(columns={'subquery_list': 'subquery'})
['subquery'] = ...
[['query_id', 'query', 'subquery']]
.reset_index(drop=True)


In [6]:
print("\n" + "=" * 70)
print("Step 3: Semantic join - find relevant docs for each subquery")
print("=" * 70)

# Build the join LazyFrame - using LazyFrame as right side
retrieval_df = exploded_df.sem_join(
    corpus_lf,  # Pass LazyFrame to sem_join
    "{subquery:left} can be answered using information from {content:right}",
    suffix="_match",
)

print("\nRetrieval LazyFrame:")
print(retrieval_df)


Step 3: Semantic join - find relevant docs for each subquery

Retrieval LazyFrame:
Pipeline('queries')
.sem_map("Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. Return ONLY a newline-separated list of subqueries, nothing else. Each subquery should be on its own line.")
['subquery_list'] = ...
.explode('subquery_list')
.reset_index(drop=True)
.rename(columns={'subquery_list': 'subquery'})
['subquery'] = ...
[['query_id', 'query', 'subquery']]
.reset_index(drop=True)
.sem_join(<Pipeline>, "{subquery:left} can be answered using information from {content:right}"):
    .add_source('corpus')


In [7]:
print("\n" + "=" * 70)
print("Step 4: Aggregate docs per original query")
print("=" * 70)


def aggregate_docs(group):
    """Aggregate retrieved documents per query."""
    unique_docs = group.drop_duplicates(subset=["doc_id"])
    docs_text = "\n\n---\n\n".join(unique_docs["content"].tolist())
    return pd.Series({"query": group["query"].iloc[0], "num_docs": len(unique_docs), "aggregated_docs": docs_text})


aggregated_df = retrieval_df.groupby("query_id").apply(aggregate_docs, include_groups=False).reset_index()

print("\nAggregated LazyFrame:")
print(aggregated_df)


Step 4: Aggregate docs per original query

Aggregated LazyFrame:
Pipeline('queries')
.sem_map("Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. Return ONLY a newline-separated list of subqueries, nothing else. Each subquery should be on its own line.")
['subquery_list'] = ...
.explode('subquery_list')
.reset_index(drop=True)
.rename(columns={'subquery_list': 'subquery'})
['subquery'] = ...
[['query_id', 'query', 'subquery']]
.reset_index(drop=True)
.sem_join(<Pipeline>, "{subquery:left} can be answered using information from {content:right}"):
    .add_source('corpus')
.groupby('query_id')
.apply(<function aggregate_docs at 0x32e21e840>, include_groups=False)
.reset_index()


In [8]:
print("\n" + "=" * 70)
print("Step 5: Answer queries using retrieved docs")
print("=" * 70)

answer_df = aggregated_df.sem_map(
    "Using the following documents:\n\n{aggregated_docs}\n\n"
    "Please provide a comprehensive answer to this question: {query}\n\n"
    "Synthesize information from all relevant documents to give a complete answer.",
    suffix="_answer",
)

print("\nAnswer generation LazyFrame:")
print(answer_df)


Step 5: Answer queries using retrieved docs

Answer generation LazyFrame:
Pipeline('queries')
.sem_map("Break down {query} into 2-3 simpler, focused subqueries that would help answer the main query. Return ONLY a newline-separated list of subqueries, nothing else. Each subquery should be on its own line.")
['subquery_list'] = ...
.explode('subquery_list')
.reset_index(drop=True)
.rename(columns={'subquery_list': 'subquery'})
['subquery'] = ...
[['query_id', 'query', 'subquery']]
.reset_index(drop=True)
.sem_join(<Pipeline>, "{subquery:left} can be answered using information from {content:right}"):
    .add_source('corpus')
.groupby('query_id')
.apply(<function aggregate_docs at 0x32e21e840>, include_groups=False)
.reset_index()
.sem_map("Using the following documents:

{aggregated_docs}

Please provide a comprehensive answer to this question: {query}

Synthesize information from all relevant documents to give a complete answer.")


In [9]:
print("\n" + "=" * 70)
print("Step 6: Execute the LazyFrame pipeline")
print("=" * 70)

# Execute the LazyFrame - this runs all the operations we've built
result_df = answer_df.execute({"queries": queries_df, "corpus": corpus_df})

print("\nExecution complete! Results:")
print(result_df)


Step 6: Execute the LazyFrame pipeline


Mapping: 100%|██████████ 2/2 LM calls [00:01<00:00,  1.55it/s]
Join comparisons: 100%|██████████ 60/60 LM Calls [00:03<00:00, 18.63it/s]
Mapping: 100%|██████████ 2/2 LM calls [00:04<00:00,  2.33s/it]


Execution complete! Results:
   query_id                                              query  num_docs  \
0         1  What are the health benefits of exercise and w...         4   
1         2  How does climate change affect biodiversity an...         5   

                                     aggregated_docs  \
0  Regular exercise improves cardiovascular healt...   
1  Climate change is causing rising temperatures ...   

                                             _answer  
0  Regular exercise offers significant health ben...  
1  Climate change significantly affects biodivers...  


In [ ]:
# View results as a list of dictionaries
result_df.to_dict(orient="records")

[{'query_id': 1,
  'query': 'What are the health benefits of exercise and what types of exercise are best for beginners?',
  'num_docs': 4,
  'aggregated_docs': "Regular exercise improves cardiovascular health by strengthening the heart muscle and improving blood circulation. It reduces the risk of heart disease and stroke.\n\n---\n\nWalking is an excellent low-impact exercise for beginners. It requires no special equipment and can be done anywhere. Start with 15-20 minutes daily.\n\n---\n\nStrength training helps build muscle mass and increases metabolism. Beginners should start with bodyweight exercises like squats and push-ups.\n\n---\n\nSwimming is a great full-body workout that's easy on the joints. It's ideal for beginners and those with mobility issues.",
  '_answer': 'Regular exercise offers significant health benefits, particularly for cardiovascular health. It strengthens the heart muscle and improves blood circulation, which helps reduce the risk of heart disease and stroke.

In [ ]:
# Alternative approach: Direct join without subquery decomposition
# This demonstrates a simpler RAG pipeline
simple_rag_df = LazyFrame("queries", queries_df).sem_join(
    corpus_lf, "{query:left} can be answered using information from {content:right}", suffix="_match"
)

# Execute
simple_result = simple_rag_df.execute({"queries": queries_df, "corpus": corpus_df})
print("Simple RAG result:")
print(simple_result)

Mapping: 100%|██████████ 2/2 LM calls [00:01<00:00,  1.09it/s]


ValueError: No DataFrame provided for right source 'corpus'. Pass inputs={'corpus': df} to execute().

In [ ]:
# Display the simple RAG result
simple_result

,query_id,query,num_docs,aggregated_docs,answer
0,1,What are the health benefits of exercise and w...,4,Regular exercise improves cardiovascular healt...,Regular exercise offers significant health ben...
1,2,How does climate change affect biodiversity an...,5,Climate change is causing rising temperatures ...,Climate change significantly affects biodivers...
